In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import KFold

In [3]:
import pandas as pd

df = pd.read_csv("../../data/processed/ml_dataset.csv")

print(df.shape)
display(df.head())
print(df.columns.tolist())

(2685, 28)


,stock_code,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,...,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20,target_return_5d
0,660,2024-06-11,0.021635,0.094233,0.054591,0.181212,0.011905,0.043689,0.009615,203000.0,...,6187.902397,5195.016563,992.885834,0.031048,0.023996,6757.468051,0.253659,3435519.80,0.893653,0.103529
1,660,2024-06-12,0.011765,0.112261,0.061728,0.169750,0.014151,0.023697,-0.002353,207340.0,...,6911.664995,5538.346249,1373.318746,0.028770,0.023814,6631.934619,-0.299278,3381585.20,0.636190,0.086047
2,660,2024-06-13,0.032558,0.146102,0.096296,0.198057,-0.017699,0.034247,0.051163,213000.0,...,7958.354609,6022.347921,1936.006687,0.026692,0.024432,6979.653575,1.685444,3532295.70,1.635559,0.069820
3,660,2024-06-14,-0.004505,0.065060,0.129280,0.145078,-0.017778,0.041667,0.013514,215700.0,...,8607.944994,6539.467336,2068.477659,0.014806,0.023386,7123.964034,-0.426854,3443697.95,0.961531,0.058824
4,660,2024-06-17,0.009050,0.072115,0.178647,0.174302,0.018265,0.047945,-0.009050,218700.0,...,9178.331251,7067.240119,2111.091132,0.013915,0.022745,7365.109460,-0.336094,3411545.25,0.644382,0.000000


['stock_code', 'trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'target_return_5d']


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2685 entries, 0 to 2684
Data columns (total 28 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   stock_code        2685 non-null   int64  
 1   trade_date        2685 non-null   str    
 2   return_1d         2685 non-null   float64
 3   return_5d         2685 non-null   float64
 4   return_10d        2685 non-null   float64
 5   return_20d        2685 non-null   float64
 6   intraday_return   2685 non-null   float64
 7   high_low_range    2685 non-null   float64
 8   gap               2685 non-null   float64
 9   sma_5             2685 non-null   float64
 10  sma_20            2685 non-null   float64
 11  sma_60            2685 non-null   float64
 12  price_to_sma_5    2685 non-null   float64
 13  price_to_sma_20   2685 non-null   float64
 14  price_to_sma_60   2685 non-null   float64
 15  rsi_14            2685 non-null   float64
 16  roc_10            2685 non-null   float64
 17  roc_20

In [5]:
target_col = "target_return_5d"

exclude_cols = [
    "stock_code",
    "trade_date",
    target_col
]

X = df.drop(columns=exclude_cols)
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

X shape: (2685, 25)
y shape: (2685,)

Features:
['return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20']


In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import KFold

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [8]:
n_features_list = [5, 10, 15, 20]
tol_list = [0, 0.001, 0.005]

In [4]:
results = []

for n_features in n_features_list:
    for tol in tol_list:

        print(f"\n{'='*60}")
        print(f"SFS + Random Forest")
        print(f"n_features_to_select = {n_features}")
        print(f"tol = {tol}")
        print(f"{'='*60}")

        sfs = SequentialFeatureSelector(
            estimator=rf,
            n_features_to_select=n_features,
            direction="forward",
            tol=tol,
            scoring="neg_mean_squared_error",
            cv=cv,
            n_jobs=-1
        )

        sfs.fit(X, y)

        selected_features = X.columns[sfs.get_support()].tolist()

        results.append({
            "n_features_to_select": n_features,
            "tol": tol,
            "n_selected_features": len(selected_features),
            "selected_features": ",".join(selected_features)
        })

        print(f"Selected features: {len(selected_features)}")
        print(selected_features)

NameError: name 'n_features_list' is not defined

In [ ]:
sfs_results = pd.DataFrame(results)

display(sfs_results)



,n_features_to_select,tol,n_selected_features,selected_features
0,5,0.000,5,"sma_60,price_to_sma_60,macd_signal,volatility_..."
1,5,0.001,5,"sma_60,price_to_sma_60,macd_signal,volatility_..."
2,5,0.005,5,"sma_60,price_to_sma_60,macd_signal,volatility_..."
3,10,0.000,10,"sma_5,sma_20,sma_60,price_to_sma_60,macd,macd_..."
4,10,0.001,10,"sma_5,sma_20,sma_60,price_to_sma_60,macd,macd_..."
5,10,0.005,10,"sma_5,sma_20,sma_60,price_to_sma_60,macd,macd_..."
6,15,0.000,15,"return_10d,sma_5,sma_20,sma_60,price_to_sma_5,..."
7,15,0.001,15,"return_10d,sma_5,sma_20,sma_60,price_to_sma_5,..."
8,15,0.005,15,"return_10d,sma_5,sma_20,sma_60,price_to_sma_5,..."
9,20,0.000,20,"return_5d,return_10d,return_20d,gap,sma_5,sma_..."


In [1]:
sfs_results.to_csv(
    "../../data/processed/filter_results/sfs_random_forest_feature_sets.csv",
    index=False
)

print("Saved successfully.")

NameError: name 'sfs_results' is not defined

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate
import numpy as np
import pandas as pd

scoring = {
    "mse": "neg_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

performance_results = []

for _, row in sfs_results.iterrows():

    n_features = row["n_features_to_select"]
    tol = row["tol"]

    features = row["selected_features"].split(",")

    X_selected = X[features]

    rf = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_validate(
        rf,
        X_selected,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    performance_results.append({
        "n_features_to_select": n_features,
        "tol": tol,
        "n_selected_features": len(features),
        "MSE": -scores["test_mse"].mean(),
        "RMSE": np.sqrt(-scores["test_mse"].mean()),
        "MAE": -scores["test_mae"].mean(),
        "R2": scores["test_r2"].mean()
    })

performance_df = pd.DataFrame(performance_results)

performance_df

NameError: name 'sfs_results' is not defined

In [2]:
#결과 저장
performance_df.to_csv(
    "../../data/processed/sfs_random_forest_performance.csv",
    index=False
)

NameError: name 'performance_df' is not defined

In [14]:
performance_df.sort_values("RMSE")

,n_features_to_select,tol,n_selected_features,MSE,RMSE,MAE,R2
3,10,0.000,10,0.002203,0.046939,0.032997,0.627828
4,10,0.001,10,0.002203,0.046939,0.032997,0.627828
5,10,0.005,10,0.002203,0.046939,0.032997,0.627828
1,5,0.001,5,0.002336,0.048336,0.033952,0.606246
0,5,0.000,5,0.002336,0.048336,0.033952,0.606246
2,5,0.005,5,0.002336,0.048336,0.033952,0.606246
7,15,0.001,15,0.002433,0.049327,0.034789,0.588933
8,15,0.005,15,0.002433,0.049327,0.034789,0.588933
6,15,0.000,15,0.002433,0.049327,0.034789,0.588933
9,20,0.000,20,0.002723,0.052179,0.037013,0.540898
